<a href="https://colab.research.google.com/github/divya-dataengineer/aws-glue-transformations/blob/main/Pyspark_Practicles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Create SparkSession


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("PySpark_Interview_Practice").getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.0.4


**Why do we need SparkSession?**


SparkSession is the entry point for working with Spark DataFrames, SQL and other Spark functionality in modern Spark applications.

#2. Load Customers.csv and Orders.csv as RAW strings

In [2]:
#At this stage, the important thing is that SignupDate and OrderDate should be a string, not a date.

#Customer Dataset
customers_df = spark.read.option("header", True).option("inferSchema", False).option("mode", "PERMISSIVE").csv("Customers.csv")
customers_df.show(20, truncate=False)
customers_df.printSchema()

#Orders Dataset
orders_df = spark.read.option("header", True).option("inferSchema", False).option("mode", "PERMISSIVE").csv("Orders.csv")
orders_df.show(20, truncate=False)
orders_df.printSchema()

+----------+---------------+---------+----------+
|CustomerID|CustomerName   |Country  |SignupDate|
+----------+---------------+---------+----------+
|1         |Arjun Smith    |USA      |1/30/2024 |
|2         |Mason Garcia   |Germany  |5/23/2022 |
|3         |Fatima Smith   |Japan    |3/31/2022 |
|4         |Henry Tanaka   |USA      |1/31/2022 |
|5         |Liam Garcia    |Germany  |6/2/2023  |
|6         |Abigail Patel  |Japan    |7/23/2022 |
|7         |Wei Kim        |France   |8/14/2022 |
|8         |James Nguyen   |Canada   |4/8/2024  |
|9         |Aarav Johnson  |France   |12/15/2022|
|10        |Mason Johnson  |Germany  |2/21/2024 |
|11        |Lucas Smith    |India    |1/25/2023 |
|12        |Sophia Dubois  |Australia|9/11/2023 |
|13        |Mason Patel    |Brazil   |7/4/2023  |
|14        |Sophia Tanaka  |India    |7/20/2023 |
|15        |Isabella Nguyen|Australia|8/15/2023 |
|16        |Ethan Smith    |USA      |11/9/2023 |
|17        |Ava Müller     |India    |5/25/2024 |


#3. CUSTOMER — preserve original SignupDate

In [3]:
#Don't overwrite the source value.

customers_df = customers_df.withColumnRenamed("SignupDate","SignupDate_Raw")
customers_df.select("CustomerID","CustomerName","Country","SignupDate_Raw").show(20, truncate=False)

+----------+---------------+---------+--------------+
|CustomerID|CustomerName   |Country  |SignupDate_Raw|
+----------+---------------+---------+--------------+
|1         |Arjun Smith    |USA      |1/30/2024     |
|2         |Mason Garcia   |Germany  |5/23/2022     |
|3         |Fatima Smith   |Japan    |3/31/2022     |
|4         |Henry Tanaka   |USA      |1/31/2022     |
|5         |Liam Garcia    |Germany  |6/2/2023      |
|6         |Abigail Patel  |Japan    |7/23/2022     |
|7         |Wei Kim        |France   |8/14/2022     |
|8         |James Nguyen   |Canada   |4/8/2024      |
|9         |Aarav Johnson  |France   |12/15/2022    |
|10        |Mason Johnson  |Germany  |2/21/2024     |
|11        |Lucas Smith    |India    |1/25/2023     |
|12        |Sophia Dubois  |Australia|9/11/2023     |
|13        |Mason Patel    |Brazil   |7/4/2023      |
|14        |Sophia Tanaka  |India    |7/20/2023     |
|15        |Isabella Nguyen|Australia|8/15/2023     |
|16        |Ethan Smith    |

# 3.1 CUSTOMER — safely convert multiple SignupDate formats

In [4]:
#Use try_to_timestamp() so a mismatch returns NULL instead of crashing the Spark job.

customers_df = customers_df.withColumn("SignupDate",
    expr("""coalesce(try_to_timestamp(trim(SignupDate_Raw),'yyyy-MM-dd'),
                     try_to_timestamp(trim(SignupDate_Raw),'M/d/yyyy'))""").cast("date"))

customers_df.select("SignupDate_Raw","SignupDate").show(50, truncate=False)

+--------------+----------+
|SignupDate_Raw|SignupDate|
+--------------+----------+
|1/30/2024     |2024-01-30|
|5/23/2022     |2022-05-23|
|3/31/2022     |2022-03-31|
|1/31/2022     |2022-01-31|
|6/2/2023      |2023-06-02|
|7/23/2022     |2022-07-23|
|8/14/2022     |2022-08-14|
|4/8/2024      |2024-04-08|
|12/15/2022    |2022-12-15|
|2/21/2024     |2024-02-21|
|1/25/2023     |2023-01-25|
|9/11/2023     |2023-09-11|
|7/4/2023      |2023-07-04|
|7/20/2023     |2023-07-20|
|8/15/2023     |2023-08-15|
|11/9/2023     |2023-11-09|
|5/25/2024     |2024-05-25|
|10/12/2022    |2022-10-12|
|1/15/2023     |2023-01-15|
|12/20/2023    |2023-12-20|
|10/13/2023    |2023-10-13|
|6/17/2022     |2022-06-17|
|10/18/2023    |2023-10-18|
|12/3/2023     |2023-12-03|
|4/21/2024     |2024-04-21|
+--------------+----------+



# 3.2 CUSTOMER — check invalid SignupDate records

In [5]:
#This is an important Data Quality check.

invalid_signup_dates = customers_df.filter(col("SignupDate").isNull()).select("SignupDate_Raw")

invalid_signup_dates.show(100, truncate=False)
print("Invalid SignupDate records:", invalid_signup_dates.count())

+--------------+
|SignupDate_Raw|
+--------------+
+--------------+

Invalid SignupDate records: 0


# 3.3 CUSTOMER — final schema check

In [6]:
customers_df.printSchema()

root
 |-- CustomerID: string (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- SignupDate_Raw: string (nullable = true)
 |-- SignupDate: date (nullable = true)



# 4. ORDERS — preserve original OrderDate

In [7]:
orders_df = orders_df.withColumnRenamed("OrderDate","OrderDate_Raw")

orders_df.select("OrderID","CustomerID","OrderDate_Raw").show(50, truncate=False)

+-------+----------+-------------+
|OrderID|CustomerID|OrderDate_Raw|
+-------+----------+-------------+
|1017   |24        |15-Apr-24    |
|1080   |8         |24-Jan-24    |
|1023   |2         |10-Jul-24    |
|1029   |6         |4-Apr-24     |
|1021   |1         |12-Feb-24    |
|1008   |25        |9-Feb-24     |
|1058   |8         |20-Jan-24    |
|1078   |18        |12-Mar-24    |
|1014   |23        |1/2/2024     |
|1022   |5         |18-Apr-24    |
|1073   |1         |18-Jan-24    |
|1027   |25        |27-Apr-24    |
|1037   |8         |6/7/2024     |
|1065   |11        |7-Mar-24     |
|1063   |24        |20-Jun-24    |
|1010   |3         |6-Jul-24     |
|1077   |11        |8-Jan-24     |
|1005   |22        |5/21/2024    |
|1064   |3         |4/10/2024    |
|1057   |21        |7/11/2024    |
|1050   |10        |6/17/2024    |
|1026   |22        |3/7/2024     |
|1053   |8         |6-May-24     |
|1042   |9         |1/8/2024     |
|1052   |15        |6/5/2024     |
|1067   |21        |

#4.1 ORDERS — safely convert ALL known date formats

In [8]:
orders_df = orders_df.withColumn("OrderDate",
    expr("""coalesce(try_to_timestamp(trim(OrderDate_Raw),'d-MMM-yy'),
                     try_to_timestamp(trim(OrderDate_Raw),'M/d/yyyy'),
                     try_to_timestamp(trim(OrderDate_Raw),'yyyy-MM-dd'))""").cast("date"))

orders_df.select("OrderDate_Raw","OrderDate").show(100, truncate=False)

+-------------+----------+
|OrderDate_Raw|OrderDate |
+-------------+----------+
|15-Apr-24    |2024-04-15|
|24-Jan-24    |2024-01-24|
|10-Jul-24    |2024-07-10|
|4-Apr-24     |2024-04-04|
|12-Feb-24    |2024-02-12|
|9-Feb-24     |2024-02-09|
|20-Jan-24    |2024-01-20|
|12-Mar-24    |2024-03-12|
|1/2/2024     |2024-01-02|
|18-Apr-24    |2024-04-18|
|18-Jan-24    |2024-01-18|
|27-Apr-24    |2024-04-27|
|6/7/2024     |2024-06-07|
|7-Mar-24     |2024-03-07|
|20-Jun-24    |2024-06-20|
|6-Jul-24     |2024-07-06|
|8-Jan-24     |2024-01-08|
|5/21/2024    |2024-05-21|
|4/10/2024    |2024-04-10|
|7/11/2024    |2024-07-11|
|6/17/2024    |2024-06-17|
|3/7/2024     |2024-03-07|
|6-May-24     |2024-05-06|
|1/8/2024     |2024-01-08|
|6/5/2024     |2024-06-05|
|3/30/2024    |2024-03-30|
|6-Jun-24     |2024-06-06|
|9-Jul-24     |2024-07-09|
|15-Jul-24    |2024-07-15|
|5/20/2024    |2024-05-20|
|21-Jun-24    |2024-06-21|
|6/24/2024    |2024-06-24|
|5/17/2024    |2024-05-17|
|5-Jun-24     |2024-06-05|
|

#4.2 ORDERS — Data Quality check

In [9]:
invalid_order_dates = orders_df.filter(col("OrderDate").isNull()).select("OrderDate_Raw")

invalid_order_dates.show(100, truncate=False)

print("Invalid OrderDate records:",invalid_order_dates.count())

+-------------+
|OrderDate_Raw|
+-------------+
+-------------+

Invalid OrderDate records: 0


#5. Now explicitly cast numeric columns

In [10]:
#Because we deliberately loaded everything as strings, we now control the schema.
#customers
customers_df = customers_df.withColumn("CustomerID",col("CustomerID").cast("integer"))

#orders
orders_df = (orders_df.withColumn("OrderID", col("OrderID").cast("integer"))
                      .withColumn("CustomerID", col("CustomerID").cast("integer"))
                      .withColumn("Price", regexp_replace(col("Price"),"[^0-9.-]","").cast("double"))
                      .withColumn("Quantity", col("Quantity").cast("integer")))

customers_df.printSchema()
orders_df.printSchema()



root
 |-- CustomerID: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- SignupDate_Raw: string (nullable = true)
 |-- SignupDate: date (nullable = true)

root
 |-- OrderID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- OrderDate_Raw: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- OrderDate: date (nullable = true)



#6. Transformations

#6.1 Verify our cleaned DataFrames

In [11]:
print("CUSTOMERS")
customers_df.show(5, truncate=False)
customers_df.printSchema()

print("\nORDERS")
orders_df.show(5, truncate=False)
orders_df.printSchema()

CUSTOMERS
+----------+------------+-------+--------------+----------+
|CustomerID|CustomerName|Country|SignupDate_Raw|SignupDate|
+----------+------------+-------+--------------+----------+
|1         |Arjun Smith |USA    |1/30/2024     |2024-01-30|
|2         |Mason Garcia|Germany|5/23/2022     |2022-05-23|
|3         |Fatima Smith|Japan  |3/31/2022     |2022-03-31|
|4         |Henry Tanaka|USA    |1/31/2022     |2022-01-31|
|5         |Liam Garcia |Germany|6/2/2023      |2023-06-02|
+----------+------------+-------+--------------+----------+
only showing top 5 rows
root
 |-- CustomerID: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- SignupDate_Raw: string (nullable = true)
 |-- SignupDate: date (nullable = true)


ORDERS
+-------+-------------+---------+------+--------+-------------+----------+----------+
|OrderID|ProductName  |Category |Price |Quantity|OrderDate_Raw|CustomerID|OrderDate |
+-------+-------------+-----

In [12]:
#get counts

print("Customers:", customers_df.count())
print("Orders:", orders_df.count())

Customers: 25
Orders: 84


#6.2 Clean the STRING columns

In [13]:
customers_df.printSchema()
orders_df.printSchema()

#For the customer data, let's clean common string fields.
customers_df = (customers_df
    .withColumn("CustomerName", trim(col("CustomerName")))
    .withColumn("Country", trim(col("Country"))))

#For Orders, adapt according to your actual columns. For example:
orders_df = (
    orders_df
    .withColumn("Category", trim(col("Category"))))

print(customers_df.columns)
print(orders_df.columns)

root
 |-- CustomerID: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- SignupDate_Raw: string (nullable = true)
 |-- SignupDate: date (nullable = true)

root
 |-- OrderID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- OrderDate_Raw: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- OrderDate: date (nullable = true)

['CustomerID', 'CustomerName', 'Country', 'SignupDate_Raw', 'SignupDate']
['OrderID', 'ProductName', 'Category', 'Price', 'Quantity', 'OrderDate_Raw', 'CustomerID', 'OrderDate']


#6.3 Normalize Category

In [14]:
orders_df = orders_df.withColumn("Category",lower(trim(col("Category"))))

orders_df.select("Category").distinct().show()

+-----------+
|   Category|
+-----------+
| stationery|
|    kitchen|
| electronic|
|  furniture|
|electronics|
| stationary|
|    fitness|
+-----------+



#6.4 Calculating Revenue

In [15]:
orders_df = orders_df.withColumn("Revenue", col("Price") * col("Quantity"))

orders_df.select(
    "OrderID",
    "CustomerID",
    "Price",
    "Quantity",
    "Revenue"
).show(10, truncate=False)

+-------+----------+------+--------+-------+
|OrderID|CustomerID|Price |Quantity|Revenue|
+-------+----------+------+--------+-------+
|1017   |24        |31.46 |4       |125.84 |
|1080   |8         |180.27|1       |180.27 |
|1023   |2         |NULL  |1       |NULL   |
|1029   |6         |233.85|2       |467.7  |
|1021   |1         |124.49|1       |124.49 |
|1008   |25        |218.67|2       |437.34 |
|1058   |8         |125.27|5       |626.35 |
|1078   |18        |59.33 |3       |177.99 |
|1014   |23        |13.09 |2       |26.18  |
|1022   |5         |232.22|2       |464.44 |
+-------+----------+------+--------+-------+
only showing top 10 rows


#6.5 Revenue data-quality checks

In [16]:
orders_df.filter(
    col("Price").isNull() |
    col("Quantity").isNull() |
    col("Revenue").isNull()
).show(truncate=False)

+-------+--------------+-----------+-----+--------+-------------+----------+----------+-------+
|OrderID|ProductName   |Category   |Price|Quantity|OrderDate_Raw|CustomerID|OrderDate |Revenue|
+-------+--------------+-----------+-----+--------+-------------+----------+----------+-------+
|1023   |Office Chair  |furniture  |NULL |1       |10-Jul-24    |2         |2024-07-10|NULL   |
|1037   |Office Chair  |furniture  |NULL |4       |6/7/2024     |8         |2024-06-07|NULL   |
|1063   |Office Chair  |furniture  |NULL |3       |20-Jun-24    |24        |2024-06-20|NULL   |
|1052   |Coffee Maker  |kitchen    |NULL |5       |6/5/2024     |15        |2024-06-05|NULL   |
|1060   |Sticky Notes  |stationery |NULL |4       |25-Mar-24    |14        |2024-03-25|NULL   |
|1052   |Coffee Maker  |kitchen    |NULL |5       |6/5/2024     |15        |2024-06-05|NULL   |
|1074   |Blender       |kitchen    |NULL |3       |11-Jun-24    |23        |2024-06-11|NULL   |
|1001   |Wireless Mouse|electronics|NULL

#6.6 Check invalid values

In [17]:
orders_df.filter(
    (col("Price") < 0) |
    (col("Quantity") <= 0) |
    (col("Revenue") < 0)
).show(truncate=False)

+-------+-----------+--------+-----+--------+-------------+----------+---------+-------+
|OrderID|ProductName|Category|Price|Quantity|OrderDate_Raw|CustomerID|OrderDate|Revenue|
+-------+-----------+--------+-----+--------+-------------+----------+---------+-------+
+-------+-----------+--------+-----+--------+-------------+----------+---------+-------+



#7. Calculate total revenue (Aggregartions)

In [18]:
from pyspark.sql.functions import sum

total_revenue = orders_df.agg(
    sum("Revenue").alias("Total_Revenue")
)

total_revenue.show()

+------------------+
|     Total_Revenue|
+------------------+
|21266.989999999998|
+------------------+



#7.1 Revenue by Customer

In [19]:
#This is a wide transformation because groupBy() requires a shuffle.

customer_revenue = (
    orders_df
    .groupBy("CustomerID")
    .agg(
        sum("Revenue").alias("Total_Revenue")
    )
    .orderBy(col("Total_Revenue").desc())
)

customer_revenue.show(10)

+----------+------------------+
|CustomerID|     Total_Revenue|
+----------+------------------+
|         8|2968.0499999999997|
|         3|2348.1499999999996|
|        18|           2224.24|
|         9|           2148.11|
|        24|           1340.88|
|        12|            990.13|
|        15|            979.65|
|        25|            908.88|
|        22| 852.6299999999999|
|        23|            636.44|
+----------+------------------+
only showing top 10 rows


#8. Join Orders + Customers

In [20]:
print("Orders columns:")
print(orders_df.columns)

print("\nCustomers columns:")
print(customers_df.columns)

orders_df.printSchema()
customers_df.printSchema()

customers_df.select(
    "CustomerID",
    "CustomerName",
    "Country"
)



Orders columns:
['OrderID', 'ProductName', 'Category', 'Price', 'Quantity', 'OrderDate_Raw', 'CustomerID', 'OrderDate', 'Revenue']

Customers columns:
['CustomerID', 'CustomerName', 'Country', 'SignupDate_Raw', 'SignupDate']
root
 |-- OrderID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- OrderDate_Raw: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- Revenue: double (nullable = true)

root
 |-- CustomerID: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- SignupDate_Raw: string (nullable = true)
 |-- SignupDate: date (nullable = true)



DataFrame[CustomerID: int, CustomerName: string, Country: string]

#8.1 Create the joined DataFrame

In [21]:
from pyspark.sql.functions import col

orders_customers_df = (
    orders_df
    .join(
        customers_df,
        orders_df["CustomerID"] == customers_df["CustomerID"],
        "inner"
    )
    .select(
        orders_df["OrderID"],
        orders_df["CustomerID"],
        orders_df["ProductName"],
        orders_df["Category"],
        orders_df["Price"],
        orders_df["Quantity"],
        orders_df["OrderDate"],
        orders_df["Revenue"],
        customers_df["CustomerName"],
        customers_df["Country"]
    )
)

orders_customers_df.show(10, truncate=False)

+-------+----------+-----------------+-----------+------+--------+----------+-------+-------------+-------+
|OrderID|CustomerID|ProductName      |Category   |Price |Quantity|OrderDate |Revenue|CustomerName |Country|
+-------+----------+-----------------+-----------+------+--------+----------+-------+-------------+-------+
|1017   |24        |Office Chair     |furniture  |31.46 |4       |2024-04-15|125.84 |Wei Kim      |Germany|
|1080   |8         |Dumbbell Set     |fitness    |180.27|1       |2024-01-24|180.27 |James Nguyen |Canada |
|1023   |2         |Office Chair     |furniture  |NULL  |1       |2024-07-10|NULL   |Mason Garcia |Germany|
|1029   |6         |Standing Desk    |furniture  |233.85|2       |2024-04-04|467.7  |Abigail Patel|Japan  |
|1021   |1         |Yoga Mat         |fitness    |124.49|1       |2024-02-12|124.49 |Arjun Smith  |USA    |
|1008   |25        |Non-stick Pan    |kitchen    |218.67|2       |2024-02-09|437.34 |Lucas Patel  |Germany|
|1058   |8         |Standing

#8.3 Validate the join

In [22]:
print("Orders:", orders_df.count())
print("Joined:", orders_customers_df.count())

Orders: 84
Joined: 84


#8.4 Find unmatched orders

In [23]:
unmatched_orders = (
    orders_df
    .join(
        customers_df,
        "CustomerID",
        "left_anti"
    )
)

unmatched_orders.show(truncate=False)

#What is left_anti?
#It returns rows from the left DataFrame that have no matching key in the right DataFrame.

+----------+-------+-----------+--------+-----+--------+-------------+---------+-------+
|CustomerID|OrderID|ProductName|Category|Price|Quantity|OrderDate_Raw|OrderDate|Revenue|
+----------+-------+-----------+--------+-----+--------+-------------+---------+-------+
+----------+-------+-----------+--------+-----+--------+-------------+---------+-------+



#8.5 Revenue by Country

In [24]:
from pyspark.sql.functions import sum

revenue_by_country = (
    orders_customers_df
    .groupBy("Country")
    .agg(
        sum("Revenue").alias("Total_Revenue")
    )
    .orderBy(
        col("Total_Revenue").desc()
    )
)

revenue_by_country.show(truncate=False)

#How would you find the top countries by revenue using PySpark?
#I first join the orders and customer datasets using CustomerID, calculate revenue at the order level, then group by Country and aggregate Revenue using sum. Finally, I order the aggregated result in descending order

+---------+------------------+
|Country  |Total_Revenue     |
+---------+------------------+
|France   |4608.68           |
|Canada   |3892.8799999999997|
|Germany  |3853.16           |
|Japan    |2977.23           |
|Australia|1969.7800000000002|
|USA      |1277.09           |
|India    |1059.3            |
|UK       |589.4             |
|Brazil   |541.74            |
|Mexico   |497.73            |
+---------+------------------+



#8.6 Reconciliation

In [25]:
#Now let's verify that the revenue hasn't mysteriously changed because of the join.

orders_df.agg(sum("Revenue").alias("Total_Revenue")).show()

orders_customers_df.agg(sum("Revenue").alias("Joined_Revenue")).show()

#After joining the fact and customer datasets, I reconcile the pre-join and post-join revenue totals to ensure the join hasn't caused record loss or duplication
# 21266.989999999998 is just normal floating-point precision. The business value is 21,266.99


+------------------+
|     Total_Revenue|
+------------------+
|21266.989999999998|
+------------------+

+------------------+
|    Joined_Revenue|
+------------------+
|21266.989999999998|
+------------------+



#9. Normal Join vs Broadcast Join

In [37]:
#Normal Join
normal_join = (orders_df.join(customers_df, "CustomerID", "inner"))
normal_join.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- BroadcastHashJoin Inner BuildRight (9)
      :- Project (4)
      :  +- Project (3)
      :     +- Filter (2)
      :        +- Scan csv  (1)
      +- BroadcastExchange (8)
         +- Project (7)
            +- Filter (6)
               +- Scan csv  (5)


(1) Scan csv 
Output [7]: [OrderID#56, ProductName#57, Category#58, Price#59, Quantity#60, OrderDate#61, CustomerID#62]
Batched: false
Location: InMemoryFileIndex [file:/content/Orders.csv]
ReadSchema: struct<OrderID:string,ProductName:string,Category:string,Price:string,Quantity:string,OrderDate:string,CustomerID:string>

(2) Filter
Input [7]: [OrderID#56, ProductName#57, Category#58, Price#59, Quantity#60, OrderDate#61, CustomerID#62]
Condition : isnotnull(cast(CustomerID#62 as int))

(3) Project
Output [8]: [cast(OrderID#56 as int) AS OrderID#166, ProductName#57, lower(trim(trim(Category#58, None), None)) AS Category#246, cast(regexp_replace(Price#59, [^0-9.-], , 1) 

In [39]:
#we are tetsing normal join automatically taking broadcast join...

print(spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

normal_join_test = (orders_df.join(customers_df, "CustomerID", "inner"))
normal_join_test.explain("formatted")

#Therefore, yes — Spark automatically selected the broadcast join because it estimated the right-side customers_df was small enough relative to the 10 MB threshold.
#Spark can automatically choose a broadcast hash join when the estimated size of a join side is below spark.sql.autoBroadcastJoinThreshold,
#provided the optimizer considers that strategy appropriate. The default threshold is commonly 10 MB

10485760b
== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- BroadcastHashJoin Inner BuildRight (9)
      :- Project (4)
      :  +- Project (3)
      :     +- Filter (2)
      :        +- Scan csv  (1)
      +- BroadcastExchange (8)
         +- Project (7)
            +- Filter (6)
               +- Scan csv  (5)


(1) Scan csv 
Output [7]: [OrderID#56, ProductName#57, Category#58, Price#59, Quantity#60, OrderDate#61, CustomerID#62]
Batched: false
Location: InMemoryFileIndex [file:/content/Orders.csv]
ReadSchema: struct<OrderID:string,ProductName:string,Category:string,Price:string,Quantity:string,OrderDate:string,CustomerID:string>

(2) Filter
Input [7]: [OrderID#56, ProductName#57, Category#58, Price#59, Quantity#60, OrderDate#61, CustomerID#62]
Condition : isnotnull(cast(CustomerID#62 as int))

(3) Project
Output [8]: [cast(OrderID#56 as int) AS OrderID#166, ProductName#57, lower(trim(trim(Category#58, None), None)) AS Category#246, cast(regexp_replace(Price#59, [^0-9

In [36]:
#Broadcast Join
from pyspark.sql.functions import broadcast
broadcast_join = (orders_df.join(broadcast(customers_df), "CustomerID", "inner"))
broadcast_join.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- BroadcastHashJoin Inner BuildRight (9)
      :- Project (4)
      :  +- Project (3)
      :     +- Filter (2)
      :        +- Scan csv  (1)
      +- BroadcastExchange (8)
         +- Project (7)
            +- Filter (6)
               +- Scan csv  (5)


(1) Scan csv 
Output [7]: [OrderID#56, ProductName#57, Category#58, Price#59, Quantity#60, OrderDate#61, CustomerID#62]
Batched: false
Location: InMemoryFileIndex [file:/content/Orders.csv]
ReadSchema: struct<OrderID:string,ProductName:string,Category:string,Price:string,Quantity:string,OrderDate:string,CustomerID:string>

(2) Filter
Input [7]: [OrderID#56, ProductName#57, Category#58, Price#59, Quantity#60, OrderDate#61, CustomerID#62]
Condition : isnotnull(cast(CustomerID#62 as int))

(3) Project
Output [8]: [cast(OrderID#56 as int) AS OrderID#166, ProductName#57, lower(trim(trim(Category#58, None), None)) AS Category#246, cast(regexp_replace(Price#59, [^0-9.-], , 1) 

#9.1 Now I want to see what happens when automatic broadcasting is not available.

In [42]:
original_threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
print(original_threshold)

#temporarily disable automatic broadcast:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",-1)
print(spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

#create the same join again, without broadcast():
no_broadcast_join = (orders_df.join(customers_df, "CustomerID", "inner"))

no_broadcast_join.explain("formatted")


-1
-1
== Physical Plan ==
AdaptiveSparkPlan (14)
+- Project (13)
   +- SortMergeJoin Inner (12)
      :- Sort (6)
      :  +- Exchange (5)
      :     +- Project (4)
      :        +- Project (3)
      :           +- Filter (2)
      :              +- Scan csv  (1)
      +- Sort (11)
         +- Exchange (10)
            +- Project (9)
               +- Filter (8)
                  +- Scan csv  (7)


(1) Scan csv 
Output [7]: [OrderID#56, ProductName#57, Category#58, Price#59, Quantity#60, OrderDate#61, CustomerID#62]
Batched: false
Location: InMemoryFileIndex [file:/content/Orders.csv]
ReadSchema: struct<OrderID:string,ProductName:string,Category:string,Price:string,Quantity:string,OrderDate:string,CustomerID:string>

(2) Filter
Input [7]: [OrderID#56, ProductName#57, Category#58, Price#59, Quantity#60, OrderDate#61, CustomerID#62]
Condition : isnotnull(cast(CustomerID#62 as int))

(3) Project
Output [8]: [cast(OrderID#56 as int) AS OrderID#166, ProductName#57, lower(trim(trim(Categor

#9.2 PySpark Notes — What happens when Broadcast Join is disabled?

**1. First, understand the normal situation**

Suppose we have:

orders_df → large table
customers_df → small table
Join key → CustomerID

When Spark decides that customers_df is small enough, it can use:

Broadcast Hash Join

Conceptually:
              Customers
             (small table)
                   |
              Broadcast
          /        |        \
       Executor  Executor  Executor
          |        |        |
       Orders    Orders    Orders
          \        |        /
             Join locally

The important point is:

Spark sends a copy of the small table to the executors, so the large orders_df does not need to be shuffled across the network for the join.

**2. What if Broadcast is disabled?**

spark.conf.set("spark.sql.autoBroadcastJoinThreshold",-1)

Do not automatically use broadcast joins.
In our experiment, it chose:

**Sort-Merge Join**

**3. What is Sort-Merge Join?**

With broadcast unavailable, Spark generally needs to bring records with the same CustomerID together.

It does this through shuffle.

For example:

Orders                      Customers
------------                ------------
CustomerID                  CustomerID
101                         101
102                         102
103                         103
101                         104
104
102

Spark needs records with the same key to end up in the same partition.
Therefore:

Orders                         Customers
   |                              |
   |                              |
   +---------- SHUFFLE ---------+
                  |
          hashpartitioning
          (CustomerID, 200)
                  |
               SORT
                  |
            MERGE / JOIN

**4. What exactly is "shuffle"?**

Shuffle = redistribution of data between partitions/executors based on a key.

For our join:
orders_df.join(customers_df, "CustomerID")

Spark needs the same CustomerID values to be colocated.
For example:

Before shuffle:

Partition 1       Partition 2       Partition 3
Customer 101      Customer 102      Customer 103
Customer 103      Customer 101      Customer 102
Customer 102      Customer 104      Customer 101

After shuffle:

Partition 1       Partition 2       Partition 3
Customer 101      Customer 102      Customer 103
Customer 101      Customer 102      Customer 103
Customer 101      Customer 102      Customer 104

The exact distribution depends on Spark's partitioning/hash function, but the principle is:

Rows with the same join key are redistributed so they can be processed together.

**5. Why do we see Exchange in explain()?**

Our physical plan showed:

Exchange
  hashpartitioning(CustomerID, 200)

Exchange = shuffle operation

So when we see:

Exchange
hashpartitioning(CustomerID, 200)

Spark is shuffling data across partitions.
The 200 means Spark is using 200 shuffle partitions in this case, controlled by:

spark.sql.shuffle.partitions
spark.conf.get("spark.sql.shuffle.partitions")

**6. Why does Sort-Merge Join need sorting?**

After the shuffle, each side is partitioned by CustomerID.

Spark then sorts the records inside each partition:

Orders
   |
Exchange
   |
Sort
   |
   +------+
          |
        Merge
          |
   +------+
   |
Sort
   |
Exchange
   |
Customers

The algorithm then merges matching keys.
Hence the name:

Sort + Merge = Sort-Merge Join

**7. Broadcast Join vs Sort-Merge Join**

This is a very important interview comparison:

| Feature                     | Broadcast Hash Join                   | Sort-Merge Join        |
| --------------------------- | ------------------------------------- | ---------------------- |
| Small table broadcast?      | ✅ Yes                                 | ❌ No                   |
| Large-side shuffle          | Usually avoided                       | ✅ Yes                  |
| Small-side shuffle          | Usually avoided                       | ✅ Yes                  |
| Sorting required?           | ❌ No                                  | ✅ Yes                  |
| Network movement            | Lower                                 | Higher                 |
| Best for                    | Large + small table                   | Large + large tables   |
| Memory requirement          | Small table must fit broadcast memory | Distributed processing |
| Physical plan clue          | `BroadcastHashJoin`                   | `SortMergeJoin`        |
| `Exchange` commonly visible | Often broadcast exchange              | Shuffle exchanges      |

And the Exchange nodes showed:
hashpartitioning(CustomerID, 200)

Therefore:

Broadcast disabled → Sort-Merge Join → Shuffle both sides → Sort → Merge

So the plan became:

AdaptiveSparkPlan
        |
   SortMergeJoin
      /      \
   Sort      Sort
    |          |
 Exchange    Exchange
    |          |
 Orders     Customers

** 9. Why is shuffle expensive?**

Shuffle is expensive because data may have to move:

Executor A
     ↓
     ↓ network
     ↓
Executor B

It involves things such as:

Network I/O
Serialization/deserialization
Disk I/O
Sorting
Memory consumption
Possible disk spilling

For very large datasets, this can become a major performance bottleneck.

That's why Spark optimization often tries to reduce unnecessary shuffles.

**Very important: Broadcast disabled ≠ Shuffle disabled**

Spark may then choose a strategy such as Sort-Merge Join, which requires shuffle.

Broadcast disabled
        ↓
Different join strategy
        ↓
Sort-Merge Join
        ↓
Shuffle

When broadcast join is disabled or the smaller table exceeds the broadcast threshold, Spark may choose a Sort-Merge Join. In a Sort-Merge Join, both datasets are typically shuffled based on the join key so that matching keys are colocated in the same partitions. Spark then sorts the data within each partition and merges the sorted records. The shuffle is represented by Exchange operators in the physical plan and can be expensive because it involves network transfer, serialization, sorting, and potentially disk spill.

                JOIN
                  |
        Is broadcast possible?
             /          \
           YES           NO
            |             |
   Broadcast Hash Join   Sort-Merge Join
            |             |
    Broadcast small DF   Shuffle both sides
            |             |
     Local hash join      Sort both sides
            |             |
          Faster          Merge matching keys
                          |
                       More shuffle

**Broadcast Hash Join**

Small table → broadcast → avoid large shuffle → hash join

**Sort-Merge Join**

Shuffle → partition by join key → sort → merge

And whenever you see Exchange in a Spark physical plan, immediately think:

"Data redistribution / shuffle is happening."





#10. Partitions → repartition() → coalesce()

In [44]:
#This is directly connected to the broadcast/shuffle topic we just finished.

#Check your current partition count
print("Current partitions:", orders_df.rdd.getNumPartitions())

#Check Spark's shuffle partition setting:
print("Shuffle partitions:",spark.conf.get("spark.sql.shuffle.partitions"))

#Understand the difference
#Partition = chunk of data processed by a task

Current partitions: 1
Shuffle partitions: 200


#10.1 repartition()

In [49]:
#orders_repartitioned = orders_df.repartition(5)
orders_repartitioned = orders_df.repartition(4)
print(orders_repartitioned.rdd.getNumPartitions())

#Repartition by a column
orders_repartitioned = orders_df.repartition(4, "CustomerID")

#This is useful when we want data distributed according to a particular key—for example, before certain joins or aggregations.

4


#10.2 coalesce()

In [55]:
orders_coalesced = orders_repartitioned.coalesce(2)
print("After coalesce:", orders_coalesced.rdd.getNumPartitions())

#Unlike repartition(), coalesce() generally avoids a full shuffle when reducing partitions.

After coalesce: 2


#10.3 Proving suffle happened?

In [56]:
orders_repartitioned.explain("formatted")

#That Exchange is the evidence that repartition(4) caused a shuffle.

== Physical Plan ==
AdaptiveSparkPlan (5)
+- Exchange (4)
   +- Project (3)
      +- Project (2)
         +- Scan csv  (1)


(1) Scan csv 
Output [7]: [OrderID#56, ProductName#57, Category#58, Price#59, Quantity#60, OrderDate#61, CustomerID#62]
Batched: false
Location: InMemoryFileIndex [file:/content/Orders.csv]
ReadSchema: struct<OrderID:string,ProductName:string,Category:string,Price:string,Quantity:string,OrderDate:string,CustomerID:string>

(2) Project
Output [8]: [cast(OrderID#56 as int) AS OrderID#166, ProductName#57, lower(trim(trim(Category#58, None), None)) AS Category#246, cast(regexp_replace(Price#59, [^0-9.-], , 1) as double) AS Price#168, cast(Quantity#60 as int) AS Quantity#169, OrderDate#61 AS OrderDate_Raw#131, cast(CustomerID#62 as int) AS CustomerID#167, cast(coalesce(gettimestamp(trim(OrderDate#61, None), d-MMM-yy, TimestampType, try_to_timestamp, Some(Etc/UTC), false), gettimestamp(trim(OrderDate#61, None), M/d/yyyy, TimestampType, try_to_timestamp, Some(Etc/UTC), 

In [57]:
orders_coalesced.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (12)
+- == Final Plan ==
   ResultQueryStage (7)
   +- Coalesce (6)
      +- ShuffleQueryStage (5), Statistics(sizeInBytes=10.0 KiB, rowCount=84)
         +- Exchange (4)
            +- * Project (3)
               +- * Project (2)
                  +- Scan csv  (1)
+- == Initial Plan ==
   Coalesce (11)
   +- Exchange (10)
      +- Project (9)
         +- Project (8)
            +- Scan csv  (1)


(1) Scan csv 
Output [7]: [OrderID#56, ProductName#57, Category#58, Price#59, Quantity#60, OrderDate#61, CustomerID#62]
Batched: false
Location: InMemoryFileIndex [file:/content/Orders.csv]
ReadSchema: struct<OrderID:string,ProductName:string,Category:string,Price:string,Quantity:string,OrderDate:string,CustomerID:string>

(2) Project [codegen id : 1]
Output [8]: [cast(OrderID#56 as int) AS OrderID#166, ProductName#57, lower(trim(trim(Category#58, None), None)) AS Category#246, cast(regexp_replace(Price#59, [^0-9.-], , 1) as double) AS Price#168, cast(Qu

repartition() is used when we want to change the number of partitions or redistribute data based on a particular key. It performs a shuffle, so it is more expensive but gives better data distribution. coalesce() is mainly used to reduce the number of partitions by combining existing partitions and typically avoids a full shuffle, so it is more efficient when we only need to reduce partitions. For example, I would use repartition(4, 'CustomerID') when I need data distributed by CustomerID, whereas I might use coalesce(2) before writing a small output to reduce the number of output files."

One-line memory trick:

🔀 Repartition = redistribute + shuffle
➖ Coalesce = reduce + usually no full shuffle

##11. Data Skew + Salting

1. What is Data Skew?

Data skew occurs when data is distributed unevenly across Spark partitions.

Partition 1 → 10,000 rows
Partition 2 → 12,000 rows
Partition 3 → 11,000 rows
Partition 4 → 5,000,000 rows  🚨

Now Partition 4 takes much longer to process.

This creates the classic straggler problem:

Task 1 → ✅ 10 sec
Task 2 → ✅ 11 sec
Task 3 → ✅ 9 sec
Task 4 → 🐌 20 min
                     ↓
              Stage waits for
              slowest task
Even though most tasks finish quickly, Spark cannot complete the stage until the slow task finishes.

2. Why does skew happen?

Common causes:

One CustomerID has disproportionately many records.
One country/category has huge numbers of records.
Highly repetitive join keys.
Events/logs concentrated around a particular key.
Joins where one or more keys are extremely frequent.

For example:
CustomerID    Records
---------------------
101             500
102             700
103             600
999         5,000,000  🚨

999 is a skewed/hot key.

3. Why is skew especially bad for joins?

orders_df.join(customers_df, "CustomerID")
If CustomerID = 999 occurs millions of times in orders_df, after shuffle Spark may send all those records toward the same partition.

Orders
   |
   | Shuffle by CustomerID
   ↓

P1 → normal
P2 → normal
P3 → normal
P4 → CustomerID 999 → millions of rows 🚨

That partition becomes huge.

It can cause:

Long-running tasks
Memory pressure
Disk spilling
Out-of-memory errors
Poor cluster utilization
Overall job slowdown

4. How do we identify skew?

There are several approaches.
Method 1 — Count records by key

For our data:
from pyspark.sql.functions import count, desc

orders_df.groupBy("CustomerID").agg(count("*").alias("record_count")).orderBy(desc("record_count")).show(20)

This tells us which customers have the most records.

Method 2 — Look at Spark UI

In a real production job, the Spark UI is extremely useful.

You might see:

Task 1 → 5 sec
Task 2 → 6 sec
Task 3 → 5 sec
Task 4 → 47 min 🚨

You can investigate:

Task duration
Input size
Shuffle read
Shuffle write
Spill
Partition distribution

This is often how skew is discovered in production.

5. How do we solve Data Skew?

There are several approaches:

① Broadcast join

If one side is genuinely small:
from pyspark.sql.functions import broadcast

orders_df.join(broadcast(customers_df),"CustomerID")
This can avoid shuffling the large side.

② Salting ⭐

When broadcasting isn't appropriate and there is a heavily skewed key, we can use salting.

The basic idea:

Instead of:

CustomerID = 999
CustomerID = 999
CustomerID = 999
CustomerID = 999
...

we artificially create additional keys:
999_0
999_1
999_2
999_3
999_4

Now the records can be distributed across multiple partitions.
Before:

Customer 999
     ↓
One huge partition 🚨


After salting:

999_0 → Partition 1
999_1 → Partition 2
999_2 → Partition 3
999_3 → Partition 4
999_4 → Partition 5

This spreads the workload.

Data skew

Uneven distribution of data across partitions, usually caused by highly frequent keys, resulting in some tasks being much larger/slower than others.

Salting

A technique used to mitigate skew by adding an artificial salt value to the skewed join key, distributing records for the hot key across multiple partitions.

What is data skew and how do you handle it?

Data skew occurs when data is distributed unevenly across Spark partitions, usually because certain keys have significantly more records than others. During a shuffle or join, the skewed key can cause one partition to become much larger, resulting in a long-running task, memory pressure, or disk spill. I first identify skew using Spark UI, task/shuffle metrics, or record counts by key. Depending on the situation, I can use a broadcast join when one side is small, enable AQE skew-join handling, or use salting for heavily skewed join keys."






#11.1 Identifying data skew

In [58]:
#Identifying data skew

from pyspark.sql.functions import count, desc

orders_df.groupBy("CustomerID").agg(count("*").alias("record_count")).orderBy(desc("record_count")).show(20)

#here we don't have any skewed data. record count is less and balanced. no huge records as explained above.

+----------+------------+
|CustomerID|record_count|
+----------+------------+
|         8|           9|
|        15|           8|
|         3|           6|
|         9|           6|
|        23|           5|
|        12|           4|
|        13|           4|
|        24|           4|
|        22|           3|
|         6|           3|
|        17|           3|
|        25|           3|
|        11|           3|
|        14|           3|
|        18|           3|
|         1|           2|
|        16|           2|
|        20|           2|
|        19|           2|
|         7|           2|
+----------+------------+
only showing top 20 rows


#11.2 Create a deliberately skewed dataset

In [61]:
#1. Create a deliberately skewed dataset
from pyspark.sql.functions import lit

# Create many copies of CustomerID = 999
skewed_orders_df = (orders_df.withColumn("CustomerID", lit(999)))

print("Rows:", skewed_orders_df.count())
skewed_orders_df.groupBy("CustomerID").count().show()

#2. Add a salt to the large/skewed side
#We'll use 5 salt buckets:

from pyspark.sql.functions import floor, rand

skewed_orders_salted = (skewed_orders_df.withColumn("salt", floor(rand() * 5)))

skewed_orders_salted.groupBy("CustomerID", "salt").count().orderBy("salt").show()

Rows: 84
+----------+-----+
|CustomerID|count|
+----------+-----+
|       999|   84|
+----------+-----+

+----------+----+-----+
|CustomerID|salt|count|
+----------+----+-----+
|       999|   0|   15|
|       999|   1|   19|
|       999|   2|   15|
|       999|   3|   16|
|       999|   4|   19|
+----------+----+-----+



#11.3 Salted join

In [62]:
#1. First create the replicated customer-side records:

from pyspark.sql.functions import col, explode, array, lit

skewed_customer = spark.createDataFrame(
    [(999, "Skewed Customer", "India")],
    ["CustomerID", "CustomerName", "Country"]
)

salted_customers_df = (
    skewed_customer
    .withColumn(
        "salt",
        explode(array(
            lit(0), lit(1), lit(2), lit(3), lit(4)
        ))
    )
)

salted_customers_df.show()

+----------+---------------+-------+----+
|CustomerID|   CustomerName|Country|salt|
+----------+---------------+-------+----+
|       999|Skewed Customer|  India|   0|
|       999|Skewed Customer|  India|   1|
|       999|Skewed Customer|  India|   2|
|       999|Skewed Customer|  India|   3|
|       999|Skewed Customer|  India|   4|
+----------+---------------+-------+----+



In [65]:
#Join Customers and Orders

salted_join = (skewed_orders_salted.join(salted_customers_df, on=["CustomerID", "salt"], how="inner"))

salted_join.select("CustomerID", "salt", "CustomerName").show()

+----------+----+---------------+
|CustomerID|salt|   CustomerName|
+----------+----+---------------+
|       999|   0|Skewed Customer|
|       999|   0|Skewed Customer|
|       999|   0|Skewed Customer|
|       999|   0|Skewed Customer|
|       999|   0|Skewed Customer|
|       999|   0|Skewed Customer|
|       999|   0|Skewed Customer|
|       999|   0|Skewed Customer|
|       999|   0|Skewed Customer|
|       999|   0|Skewed Customer|
|       999|   0|Skewed Customer|
|       999|   0|Skewed Customer|
|       999|   0|Skewed Customer|
|       999|   0|Skewed Customer|
|       999|   0|Skewed Customer|
|       999|   1|Skewed Customer|
|       999|   1|Skewed Customer|
|       999|   1|Skewed Customer|
|       999|   1|Skewed Customer|
|       999|   1|Skewed Customer|
+----------+----+---------------+
only showing top 20 rows


#11.4 Complete Salting Process

            CustomerID = 999
                    +
                  salt
                    ↓
        ┌───────────┼───────────┐
       999_0       999_1      ... 999_4
         ↓           ↓              ↓
    Partition A  Partition B   Partition E
         ↓           ↓              ↓
       Orders      Orders          Orders
         └───────────┼──────────────┘
                     ↓
                  JOIN ✅


1. Identify hot key
       ↓
2. Add salt to large/skewed side
       ↓
3. Replicate matching small-side rows
       ↓
4. Join on (original_key + salt)
       ↓
5. Hot key distributed across multiple partitions

              
----------------------------------------------------------




#12. cache() vs persist()

In [69]:
#1. What is cache()?
#Spark transformations are lazy.

clean_orders = (orders_df.filter(col("Quantity") > 0).withColumn("Revenue", col("Price") * col("Quantity")))

#If we use clean_orders multiple times
clean_orders.count()
clean_orders.groupBy("CustomerID").sum("Revenue").show()
clean_orders.filter(col("Revenue") > 500).show()

#Spark may have to recompute the transformation lineage for each action.
#cache() tells Spark:
#Keep the computed DataFrame so I can reuse it without recomputing the entire lineage each time.

clean_orders.cache()
clean_orders.count()

+----------+------------------+
|CustomerID|      sum(Revenue)|
+----------+------------------+
|        12|            990.13|
|        22| 852.6299999999999|
|         1|            400.55|
|        13|            541.74|
|         6| 629.0799999999999|
|        16|             530.6|
|         3|2348.1499999999996|
|        20|            288.39|
|         5|            464.44|
|        19|             589.4|
|        15|            979.65|
|         9|           2148.11|
|        17|            169.05|
|         4|            345.94|
|         8|2968.0499999999997|
|        23|            636.44|
|         7|236.32999999999998|
|        10|            286.33|
|        25|            908.88|
|        24|           1340.88|
+----------+------------------+
only showing top 20 rows
+-------+-----------------+-----------+------+--------+-------------+----------+----------+-----------------+
|OrderID|      ProductName|   Category| Price|Quantity|OrderDate_Raw|CustomerID| OrderDate|      

84

In [70]:
#2. What is persist()?
#persist() does essentially the same caching concept, but gives you control over the storage level.

from pyspark import StorageLevel

clean_orders.persist(StorageLevel.MEMORY_AND_DISK)
clean_orders.count()

#This means Spark can keep the cached partitions in memory and use disk when necessary.


84

cache()
   ↓
uses default storage level

persist()
   ↓
you choose storage level
   ↓
MEMORY_ONLY
MEMORY_AND_DISK
DISK_ONLY
etc.

You should consider caching when:

The DataFrame is expensive to compute.
The DataFrame will be reused multiple times.
The cached data is reasonably sized.
Recomputing it would be more expensive than storing it.

Avoid unnecessary caching when:

Data is used only once.
DataFrame is very large and won't fit reasonably in available storage.
There is plenty of memory pressure.
The computation is cheap.

What is the difference between cache and persist?


Both are used to persist computed data for reuse. cache() uses Spark's default storage level, while persist() allows us to explicitly select a storage level such as memory-only or memory-and-disk. So I use cache() when the default behavior is sufficient and persist() when I need more control over storage.

In [77]:
from pyspark.sql.functions import col

clean_orders = (orders_df.filter(col("Quantity") > 0).withColumn("Revenue_Check",col("Price") * col("Quantity")))

clean_orders.cache()
clean_orders.count() # Action → materializes the cache

print(clean_orders.storageLevel)

clean_orders.cache()
print(clean_orders.storageLevel)

clean_orders.unpersist() #I don't need this cached data anymore; release the cached storage.


Disk Memory Deserialized 1x Replicated
Disk Memory Deserialized 1x Replicated


DataFrame[OrderID: int, ProductName: string, Category: string, Price: double, Quantity: int, OrderDate_Raw: string, CustomerID: int, OrderDate: date, Revenue: double, Revenue_Check: double]

#12. explian(), Check partitions, Check AQE

In [78]:
#1. explian()
orders_customers_df.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (14)
+- Project (13)
   +- SortMergeJoin Inner (12)
      :- Sort (6)
      :  +- Exchange (5)
      :     +- Project (4)
      :        +- Project (3)
      :           +- Filter (2)
      :              +- Scan csv  (1)
      +- Sort (11)
         +- Exchange (10)
            +- Project (9)
               +- Filter (8)
                  +- Scan csv  (7)


(1) Scan csv 
Output [7]: [OrderID#56, ProductName#57, Category#58, Price#59, Quantity#60, OrderDate#61, CustomerID#62]
Batched: false
Location: InMemoryFileIndex [file:/content/Orders.csv]
ReadSchema: struct<OrderID:string,ProductName:string,Category:string,Price:string,Quantity:string,OrderDate:string,CustomerID:string>

(2) Filter
Input [7]: [OrderID#56, ProductName#57, Category#58, Price#59, Quantity#60, OrderDate#61, CustomerID#62]
Condition : isnotnull(cast(CustomerID#62 as int))

(3) Project
Output [7]: [cast(OrderID#56 as int) AS OrderID#166, ProductName#57, lower(trim(trim(Category#58, 

In [79]:
#2. Check Partitions

print("Orders partitions:", orders_df.rdd.getNumPartitions())
print("Customers partitions:", customers_df.rdd.getNumPartitions())
print("Joined partitions:", orders_customers_df.rdd.getNumPartitions())

Orders partitions: 1
Customers partitions: 1
Joined partitions: 1


In [80]:
#3. Check AQE

print("AQE:",spark.conf.get("spark.sql.adaptive.enabled"))

print("Shuffle partitions:",spark.conf.get("spark.sql.shuffle.partitions"))

AQE: true
Shuffle partitions: 200
